# 11 - Environment probe (~20 seconds, no install)

One question: **can anything on this image run the cp312 wheelhouse?**

Read the three verdict lines at the bottom. Nothing is installed and the GPU is
only reported, so this is safe to re-run whenever Kaggle changes an image.

In [ ]:
import glob
import json
import os
import re
import subprocess
import sys
from pathlib import Path

def out(msg=''):
    print(str(msg), flush=True)

out('=== this interpreter ===')
out('sys.version   ' + sys.version.replace('\n', ' '))
out('sys.executable ' + sys.executable)
out('tag           cp' + str(sys.version_info.major) + str(sys.version_info.minor))

out('')
out('=== image identity ===')
for k in sorted(os.environ):
    if 'DOCKER' in k.upper() or 'KAGGLE' in k.upper() or 'IMAGE' in k.upper():
        out('  ' + k + '=' + str(os.environ[k])[:160])
for f in ('/etc/os-release',):
    if Path(f).is_file():
        for line in Path(f).read_text().splitlines():
            if line.startswith(('PRETTY_NAME', 'VERSION_ID')):
                out('  ' + line)

out('')
out('=== other interpreters on this image ===')
# A 3.12 interpreter anywhere would let the cp312 wheelhouse install into it,
# which would restore the measuring rig without waiting on the organisers.
cands = set()
for pat in ('/usr/bin/python3.*', '/usr/local/bin/python3.*', '/opt/conda/bin/python3.*',
            '/opt/conda/envs/*/bin/python', '/usr/local/envs/*/bin/python'):
    cands.update(glob.glob(pat))
found = {}
for c in sorted(cands):
    if c.endswith(('-config', 'm')):
        continue
    try:
        r = subprocess.run([c, '-c', 'import sys;print("%d.%d" % sys.version_info[:2])'],
                           capture_output=True, text=True, timeout=30)
        v = (r.stdout or '').strip()
        if v:
            found[c] = v
            out('  ' + c.ljust(44) + ' -> ' + v)
    except Exception as exc:
        out('  ' + c.ljust(44) + ' -> ' + type(exc).__name__)
if not found:
    out('  (none found beyond this one)')

out('')
out('=== wheelhouse ===')
INPUT = Path('/kaggle/input')
wh = None
for p in INPUT.rglob('*'):
    if p.is_dir() and 'wheelhouse' in p.name.lower() and any(p.glob('*.whl')):
        wh = p
        break
if wh is None:
    out('  WHEELHOUSE NOT ATTACHED')
    tags, n = set(), 0
else:
    out('  ' + str(wh))
    whls = sorted(wh.glob('*.whl'))
    n = len(whls)
    tags = set()
    for w in whls:
        tags.update(re.findall(r'-(cp3\d+)-', w.name))
    out('  wheels ' + str(n) + '   compiled tags ' + str(sorted(tags)))
    # The ones that actually matter: if vLLM and torch are pure-python or abi3,
    # the cp tag of the rest may not be fatal.
    for key in ('vllm', 'torch-', 'xformers', 'flashinfer_python', 'apache_tvm_ffi'):
        hit = [w.name for w in whls if w.name.lower().startswith(key)]
        out('    ' + key.ljust(18) + (hit[0] if hit else '(absent)'))

out('')
out('=== VERDICTS ===')
running = 'cp' + str(sys.version_info.major) + str(sys.version_info.minor)
v1 = running in tags if tags else None
out('1. this interpreter matches the wheelhouse: ' + str(v1) + '  (' + running + ' in ' + str(sorted(tags)) + ')')
alt312 = [c for c, v in found.items() if v == '3.12']
out('2. a 3.12 interpreter exists on this image: ' + ('YES -> ' + ', '.join(alt312) if alt312 else 'NO'))
out('3. wheelhouse has any cp313 wheel:          ' + ('YES' if 'cp313' in tags else 'NO'))
out('')
if alt312:
    out('ACTION: install the wheelhouse into ' + alt312[0] + ' and run session_runner with it.')
elif 'cp313' in tags:
    out('ACTION: the wheelhouse now covers cp313; re-run the eval kernel unchanged.')
else:
    out('ACTION: no local fix. Either the organisers must ship cp313 wheels, or we')
    out('        vendor a cp313 vLLM stack ourselves (multi-GB), or we wait.')

out('')
out('=== GPU (reported only) ===')
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                    '--format=csv,noheader'], capture_output=True, text=True)
out((r.stdout or '(nvidia-smi returned nothing)').strip())

# ---------------------------------------------------------------------------
# Part 2: is the cp312 gate actually fatal, or just blunt?
#
# vllm ships as cp38-abi3, which installs on ANY Python >= 3.8, and torch is NOT
# in the wheelhouse at all -- it comes from the image, where it is built for the
# image's own Python. So the preflight assert, which fails the whole run when the
# interpreter tag is not in the wheelhouse, may be condemning a run over a
# handful of genuinely version-locked wheels. This measures exactly which ones
# they are and whether vllm still imports without them.
out('')
out('=== cp312-locked wheels (the actual blockers) ===')
blockers, installable = [], []
if wh is not None:
    for w in sorted(wh.glob('*.whl')):
        n = w.name
        if re.search(r'-cp3\d+-cp3\d+', n) and running not in n:
            blockers.append(n)
        else:
            installable.append(n)
for b in blockers:
    out('  BLOCKED  ' + b)
out('  ' + str(len(blockers)) + ' blocked, ' + str(len(installable)) + ' installable on ' + running)

out('')
out('=== does /usr/bin/python3.12 have a usable torch? ===')
# If it does, the clean fix is to run the whole eval under 3.12. If it does not,
# that path needs a multi-GB torch+CUDA install and is not clean at all.
if alt312:
    probe_src = ('import importlib.util as u;'
                 'print("torch", u.find_spec("torch") is not None);'
                 'print("pip", u.find_spec("pip") is not None)')
    r = subprocess.run([alt312[0], '-c', probe_src], capture_output=True, text=True)
    out('  ' + ((r.stdout or '') + (r.stderr or '')).strip()[:300])
else:
    out('  no 3.12 interpreter')

out('')
out('=== try: install the installable wheels on this interpreter, then import vllm ===')
# Bounded and reversible-ish: --no-deps so nothing in the image is upgraded, and
# we only learn whether the import works. Nothing is run on the GPU.
if wh is not None and installable:
    tmp = Path('/tmp/wh_ok')
    tmp.mkdir(parents=True, exist_ok=True)
    paths = []
    for n in installable:
        if 'cutlass' in n.lower():
            continue
        src = wh / n
        dst = tmp / n
        if not dst.exists():
            try:
                os.symlink(src, dst)
            except OSError:
                pass
        paths.append(str(dst))
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *paths],
                       capture_output=True, text=True)
    out('  pip rc ' + str(r.returncode))
    if r.returncode != 0:
        out('  ' + (r.stderr or '')[-1200:])
    # `import vllm` is NOT sufficient and this is the trap that cost a run:
    # vllm/_C.abi3.so loads lazily, so the top-level import succeeds and the
    # server then dies with
    #   undefined symbol: _ZN3c1013MessageLoggerC1EPKciib
    # That is a libtorch C++ symbol. abi3 pins the PYTHON abi, not the C++ one, so
    # a wheel built against one torch will not link against another. Load _C
    # explicitly -- it is the cheapest check that reaches the real failure.
    chk = subprocess.run([sys.executable, '-c',
                          'import torch;print("torch", torch.__version__, "cuda", torch.cuda.is_available());'
                          'import vllm;print("vllm", vllm.__version__);'
                          'import vllm._C;print("vllm._C OK")'],
                         capture_output=True, text=True)
    out('  deep import check (torch, vllm, vllm._C):')
    out('    ' + ((chk.stdout or '') + (chk.stderr or '')).strip()[-1800:])
    out('')
    out('VERDICT 4: vllm._C links against this image torch: '
        + ('YES -- the rig works' if chk.returncode == 0 else 'NO -- C++ ABI mismatch'))

    # What torch was the wheelhouse's vllm actually built for? This is the number
    # that decides whether the rig is recoverable at all, and it is written in the
    # wheel's own metadata.
    out('')
    out('=== what torch does the wheelhouse vllm require? ===')
    import zipfile
    vw = [w for w in sorted(wh.glob('*.whl')) if w.name.lower().startswith('vllm-')]
    if vw:
        z = zipfile.ZipFile(vw[0])
        meta = [n for n in z.namelist() if n.endswith('METADATA')]
        if meta:
            txt = z.read(meta[0]).decode('utf-8', 'replace')
            for line in txt.splitlines():
                low = line.lower()
                if low.startswith('requires-dist') and any(
                        k in low for k in ('torch', 'nvidia', 'xformers', 'flashinfer', 'triton')):
                    out('  ' + line)
                elif low.startswith(('name:', 'version:', 'requires-python')):
                    out('  ' + line)
    else:
        out('  no vllm wheel found')

    out('')
    out('=== is that torch present anywhere on this image? ===')
    for c in sorted(set(list(found) + [sys.executable])):
        r2 = subprocess.run([c, '-c', 'import torch;print(torch.__version__)'],
                            capture_output=True, text=True)
        out('  ' + c.ljust(34) + ' -> ' + ((r2.stdout or r2.stderr or '').strip()[:80] or 'no torch'))
else:
    out('  skipped')